# 📝 [시험] 소득세법 멀티 RAG 챗봇



# 🎓 소득세법 멀티 RAG 챗봇



---
## 📝 노트북은 "시험지"입니다

```
 ┌──────────────────────────────────────────────┐
 │  모든 코드 셀이 비어 있습니다                 │
 │  # [문제 N] 주석의 요구사항대로 직접 작성하세요 │
 └──────────────────────────────────────────────┘
```

**진행 방법**
1. 셀을 위에서부터 순서대로 풉니다 (앞 문제 결과를 뒤에서 재사용)
2. 각 셀의 `요구사항` 주석을 읽고 주석 아래에 코드를 작성
3. `▶ 출력 예시`와 비교해 결과 확인
4. 막히면 바로 아래 **"이 셀이 한 일"** 설명을 다시 읽어보세요

**문제 수**: 총 25문제 (코드 셀 25개)


---

# 1️⃣ 단계 — 준비 (설치 + 키)

### 셀 1 — 설치

In [ ]:
# [문제 1] 패키지 설치
# ------------------------------------------------------------
# 요구사항
#   - ! pip install -q 로 아래 패키지를 한 줄에 설치하세요.
#       langchain, langchain-openai, langchain-community, langchain-text-splitters,
#       langgraph, pypdf, faiss-cpu, pdfplumber
#     (pdfplumber 는 문제 9의 세율표 이미지 탐색에 필요)
#   - 설치 후 "✅ 설치 완료" 출력
# ------------------------------------------------------------

**이 셀이 한 일**: 필요한 패키지를 설치했습니다.
(벡터DB는 FAISS 하나로 통일 — 추가 설치 불필요)

### 셀 2 — OpenAI 키

In [ ]:
# [문제 2] OpenAI 키 등록
# ------------------------------------------------------------
# 요구사항
#   - os 를 import
#   - try 블록: google.colab 의 userdata 로 "OPENAI_API_KEY" 를 읽어 os.environ 에 등록
#   - except Exception 블록: dotenv 의 load_dotenv(".env", override=True) 호출
#   - key = os.environ.get("OPENAI_API_KEY", "")
#   - key 가 "sk-" 로 시작하고(startswith) ASCII 문자로만 구성(isascii)되면
#       "✅ 키 준비됨", 아니면 "❌ 키 확인 필요" 출력
#   ⚠️ 키 문자열을 코드에 직접 쓰지 마세요.
# ------------------------------------------------------------

### 셀 3 — 모델 준비

In [ ]:
# [문제 3] 모델 준비 — 답변용 LLM + 검색용 임베딩
# ------------------------------------------------------------
# 요구사항
#   - langchain_openai 에서 ChatOpenAI, OpenAIEmbeddings 를 import
#   (1) 답변용 LLM → 변수 llm
#       - 모델 "gpt-4o-mini", temperature=0 (일관된 답)
#   (2) 검색용 임베딩 → 변수 embeddings
#       - 모델 "text-embedding-3-small"
#   - "✅ LLM + 임베딩 준비 완료" 출력
# ------------------------------------------------------------

# 2️⃣ 단계 — 데이터 (소득세법 PDF 로드·분할)

실제 소득세법 PDF를 불러와 조문 단위로 나눕니다.

### 셀 4 — PDF 업로드

In [ ]:
# [문제 4] PDF 파일 경로 지정
# ------------------------------------------------------------
# 요구사항
#   - 변수 PDF 에 소득세법 PDF 경로를 저장하세요.  (예: "content/소득세법_실습데이터.pdf")
#   - (선택) Colab 이라면 google.colab 의 files.upload() 로 업로드한 뒤
#       반환 딕셔너리 {파일명: bytes} 의 key 중 ".pdf" 로 끝나는 첫 파일명을 PDF 에 저장
#       (next() + 제너레이터 표현식, .lower().endswith(".pdf") 활용)
#   - f"✅ 업로드: {PDF}" 출력
# ------------------------------------------------------------

### 셀 5 — 텍스트 추출

In [ ]:
# [문제 5] PDF 텍스트 추출 + 본문 시작점 찾기
# ------------------------------------------------------------
# 요구사항
#   - pypdf, re 를 import
#   - pypdf.PdfReader(PDF) 로 PDF 를 열어 변수 reader 에 저장
#   (1) 리스트 컴프리헨션으로 reader.pages 를 돌며 각 페이지의 .extract_text() 결과를 모음 → pages
#       ※ 추출 결과가 None 일 수 있으므로 (… or "") 로 빈 문자열 처리
#   (2) pages 를 "\n" 으로 이어붙여 전체 텍스트 → full
#   (3) "총 페이지: N / 추출 텍스트 X,XXX 자" 형식으로 출력 (천 단위 콤마 :, 사용)
#   (4) 앞쪽 목차를 건너뛰기 위해 본문 시작 "제1조(목적) 이 법은" 위치를 정규식으로 찾기 → m
#       - re.search 사용, 공백이 있을 수도 없을 수도 있으므로 \s? / \s* 활용
#       - 괄호 ( ) 는 정규식에서 이스케이프 필요
#   (5) m 을 찾았으면 full[m.start():] , 못 찾으면 full 전체 → body  (안전장치)
#   (6) 본문 시작 위치, body 앞 150자 미리보기 출력
# ▶ 출력 예시:
#   총 페이지: NN / 추출 텍스트 NNN,NNN 자
#   본문 시작 위치: NNNN
#   제1조(목적) 이 법은 개인의 소득에 대하여 ...
# ------------------------------------------------------------

### 셀 6 — 조문 단위로 분할

법령은 "제○조" 단위로 나누는 게 자연스럽습니다.
(조문 = 의미 있는 단위)

In [ ]:
# [문제 6] 조문 제목 정규식 테스트
# ------------------------------------------------------------
# 요구사항
#   - "제○조(제목)" 및 "제○조의○(제목)" 를 잡는 정규식을 re.compile 로 만들어 pat 에 저장
#       그룹1: 조 번호  예) "제55조", "제 55 조", "제22조의2"   (사이 공백 \s? 허용, "의 숫자"는 선택 그룹 (?: … )?)
#       그룹2: 괄호 안 제목  예) "세율", "퇴직소득"   (괄호 안 앞뒤 공백 허용, 닫는 괄호 전까지 [^)]+? )
#   - 테스트 문자열 3개를 for 문으로 검사하여 (그룹1, 그룹2) 또는 "매칭없음" 출력
#       "제55조(세율)", "제 55 조 ( 세율)", "제22조의2(퇴직소득)"
#   - 출력 정렬: f"{text:20}->"
# ▶ 출력 예시:
#   제55조(세율)            -> ('제55조', '세율')
#   제 55 조 ( 세율)        -> ('제 55 조', '세율')
#   제22조의2(퇴직소득)     -> ('제22조의2', '퇴직소득')
# ------------------------------------------------------------

In [ ]:
# [문제 7] 본문을 "제○조" 단위로 분할 + 메타데이터
# ------------------------------------------------------------
# 요구사항
#   - langchain_core.documents 에서 Document import
#   - 문제 6의 정규식 pat 으로 body 에서 모든 매칭을 찾아 리스트로 저장 → matches  (finditer 사용)
#   - 빈 리스트 docs 생성
#   - enumerate(matches) 로 순회하며
#       jo    : 그룹1 에서 모든 공백 제거   ("제 50 조" → "제50조")  re.sub(r"\s+", "", …)
#       title : 그룹2 에서 모든 공백 제거
#       start : 현재 매칭의 시작 위치
#       end   : 다음 매칭의 시작 위치 (마지막 조문이면 len(body))
#       content : body[start:end] 를 strip()
#       - content 길이가 15자 미만이면 목차로 보고 continue (제외)
#       - Document 를 만들어 docs 에 append
#           page_content = content 앞 1500자
#           metadata     = {"조": jo, "제목": title}
#   - f"✅ 조문 {개수}개로 분할" 출력, docs[2].metadata 예시 출력
# ▶ 출력 예시:
#   ✅ 조문 NNN개로 분할
#   예시: {'조': '제3조', '제목': '과세소득의범위'}
# ------------------------------------------------------------

# 3️⃣ 단계 — 검색 준비 (임베딩 + FAISS)

조문들을 검색 가능한 벡터로 만들어 저장합니다.

### 셀 7 — 벡터DB + 검색기

In [ ]:
# [문제 8] 벡터DB(FAISS) + 검색기
# ------------------------------------------------------------
# 요구사항
#   - langchain_community.vectorstores 에서 FAISS import
#   (1) docs 와 embeddings 로 FAISS 벡터DB 생성 → 변수 vectordb   (클래스 메서드 from_documents)
#   (2) vectordb 로 검색기 생성, 질문당 3개 반환 → 변수 retriever   (as_retriever, search_kwargs 의 k)
#   - "✅ 검색 준비 완료" 출력
#   - retriever.invoke("기본공제는 얼마인가요?") 결과를 순회하며
#       "  [조번호 제목]" 형식으로 metadata 출력
# ▶ 출력 예시:
#   ✅ 검색 준비 완료
#   질문 '기본공제' 검색 결과:
#     [제50조 기본공제]
#     ...
# ------------------------------------------------------------

### 🔄 [멀티모달 보완] 세율표 이미지를 비전으로 복원

> ⚠️ 문제 발견: 소득세법 **제55조 세율표는 PDF에 '이미지'로**
> 들어 있습니다 (p51). `pypdf` 텍스트 추출로는 **안 잡힙니다.**
> 그래서 "과세표준은 어떻게 돼?" 질문에 법령봇이 제대로
> 답하지 못하고, 계산봇도 세율을 모릅니다.

```
 [텍스트 추출만]              [멀티모달 보완]
 세율표 이미지 → 0글자        비전 모델이 표를 읽어
 "확인 안 됨" ❌               텍스트로 복원 → 색인 ✅
```

**해결**: 세율표 이미지 페이지를 찾아 비전 모델(GPT-4o)로
표를 텍스트로 복원 → 벡터DB에 추가. 

In [ ]:
# [문제 9] 제55조 세율표 이미지 페이지 찾기 + 캡처
# ------------------------------------------------------------
# 요구사항
#   - base64, pdfplumber 를 import, langchain_core.messages 에서 HumanMessage import
#   (1) 세율표 페이지 자동 탐색 → 변수 rate_page (초기값 None)
#       - pdfplumber.open(PDF) 를 with 문으로 열고 pdf.pages 를 enumerate
#       - 페이지 텍스트(extract_text() or "")에 "제55조(세율)" 과 "종합소득과세표준" 이 모두 있으면
#           현재 페이지 i 와 다음 페이지 i+1 을 후보로 검사
#           후보 페이지의 .images 중 width > 200 이고 height > 100 인 '큰 이미지'가 있으면
#           rate_page = 후보 번호 로 정하고 반복 종료(break)
#       - 찾았으면 바깥 반복도 종료
#   - f"✅ 세율표 이미지 페이지 발견: p{rate_page}" 출력
#   (2) 해당 페이지를 해상도 300 이미지로 변환(to_image) 해 "rate_table.png" 로 저장
#   - "✅ 세율표 페이지 캡처 완료 (rate_table.png)" 출력
# ------------------------------------------------------------

In [ ]:
# [문제 10] 비전 모델로 세율표 이미지 → 텍스트(OCR)
# ------------------------------------------------------------
# 요구사항
#   - "rate_table.png" 를 바이너리("rb")로 읽어 base64 인코딩 후 .decode() → 변수 img_b64
#   (1) 비전 모델 → 변수 vision : ChatOpenAI, 모델 "gpt-4o" (mini 아님), temperature=0
#   (2) HumanMessage(content=[...]) 로 멀티모달 메시지 작성 → 변수 vision_msg
#       - 첫 요소 {"type": "text", "text": 프롬프트}
#           프롬프트는 "OCR 작업"임을 명확히: 표의 글자·숫자를 보이는 그대로 마크다운 표로 전사,
#           해석·조언 금지  (세무 상담처럼 쓰면 모델이 거부할 수 있음)
#       - 둘째 요소 {"type": "image_url", "image_url": {"url": data URL, "detail": "high"}}
#           data URL 형식: "data:image/png;base64,<img_b64>"
#   (3) vision.invoke([vision_msg]).content → 변수 resp
#   (4) 거부 응답 처리: resp 소문자에 "can't assist" 또는 "sorry" 가 있으면
#       영어 프롬프트("Transcribe the table in this image into a markdown table ...")로
#       같은 구조의 메시지를 만들어 1회 재시도하고 resp 갱신
#   (5) rate_table_text = resp 로 저장하고 출력
#   (6) 결과에 "sorry" / "can't" 가 남아 있으면 "⚠️ 비전 복원 실패…", 아니면 "✅ 세율표 복원 성공" 출력
# ▶ 출력 예시: | 종합소득 과세표준 | 세율 | ... 형태의 마크다운 표
# ------------------------------------------------------------

In [ ]:
# [문제 11] 복원한 세율표를 벡터DB에 추가
# ------------------------------------------------------------
# 요구사항
#   - Document 로 세율표 문서 생성 → 변수 rate_doc
#       page_content = "[소득세법 제55조 종합소득세 세율표 - 이미지에서 비전 복원]\n" + rate_table_text
#       metadata     = {"조": "제55조", "제목": "세율"}
#   - 기존 vectordb 에 rate_doc 를 '추가' 하세요 (재색인 X, add_documents 에 리스트로 전달)
#     ※ 변수 이름 주의: 문제 8에서 만든 vectordb
#   - "✅ 세율표를 벡터DB에 추가 ..." 출력
#   - retriever.invoke("종합소득세 과세표준 세율은 어떻게 되나요?") → hit
#   - hit 앞 2개를 "  [조 제목] 본문앞50자..." 형식으로 출력
# ▶ 출력 예시: [제55조 세율] [소득세법 제55조 종합소득세 세율표 ...
# ------------------------------------------------------------

# 4️⃣ 단계 — 단순 RAG (베이스라인)

먼저 가장 단순한 RAG를 만듭니다. **검색 → 생성**.
나중에 완성본과 비교할 기준선입니다.

### 셀 8 — 단순 RAG 함수

In [ ]:
# [문제 12] 단순 RAG 함수 simple_rag() — 검색 → 생성
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: simple_rag(question) -> str
#   (1) retriever 로 question 관련 조문 검색 → docs
#   (2) 검색 결과를 "[조(제목)] 본문" 형식으로 만들어 "\n\n" 으로 이어붙임 → context
#   (3) llm.invoke(프롬프트).content 로 답변 생성 → answer
#       프롬프트 구성:
#         - "소득세법 안내입니다. 아래 [자료]만 근거로 답하고,
#            없으면 '자료에서 확인되지 않습니다'라고 하세요."
#         - [자료] + context
#         - 질문: question
#   (4) answer 반환
#   - 테스트: "소득세법의 목적은 무엇인가요?" 로 호출해 "🙋 …" / "🤖 …" 형식으로 출력
# ------------------------------------------------------------

**이 셀이 한 일**: 단순 RAG를 만들었습니다.
검색 → 생성, 2단계 로직입니다. 이게 **베이스라인**입니다.

### 셀 9 — 단순 RAG의 한계 측정 (중요!)

일부러 3종 질문을 던져 어디서 약한지 직접 봅니다.
(방법론 교안의 '④ 한계 측정' 단계)

In [ ]:
# [문제 13] 단순 RAG 의 한계 측정
# ------------------------------------------------------------
# 요구사항
#   - simple_rag() 에 아래 3종 질문을 각각 던지고 "🙋 질문" / "🤖 답변" 형식으로 출력
#       ① 잡담 : "안녕하세요"
#       ② 계산 : "과세표준 2000만원이면 6% 세율로 세금은 얼마야?"
#       ③ 법령 : "기본공제는 1인당 얼마인가요?"
#   - 각 결과 사이에 구분 출력("① 잡담:", "② 계산:", "③ 법령:")
#   - 실행 후 어떤 질문에서 답이 어색/부정확한지 아래 주석으로 관찰 결과를 적으세요.
#   # 관찰: 
# ------------------------------------------------------------

**이 셀이 한 일**: 단순 RAG에 3종 질문을
던졌습니다.

> ⚠️ 관찰: ① "안녕"에도 조문 검색(어색) ② 계산은
> LLM이 틀릴 수 있음 ③ 법령 질문만 잘 답함.
> → 이 한계를 5~6단계에서 **3봇 + 라우터**로 해결합니다.

# 5️⃣ 단계 — 3개 봇 만들기

질문이 3종(계산·법령·잡담)이니 봇도 3개로 나눕니다.

### 셀 10 — 봇 ①: 계산봇 

세금 계산은 LLM이 틀리기 쉬우니, AI가 식만 뽑고
**파이썬이 직접 계산**합니다.

In [ ]:
# [문제 14] 봇 ① 계산봇 calc_bot()
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: calc_bot(question) -> str
#   - 세율을 코드에 하드코딩하지 말고, 문제 10의 rate_table_text 를 프롬프트에 넣어 근거로 사용
#   - llm.invoke(프롬프트).content → answer
#     프롬프트 구성:
#       - 역할: "당신은 세무 계산기입니다. 아래는 소득세법 제55조 실제 세율표입니다."
#       - "[종합소득세 세율표]\n" + rate_table_text 의 '실제 값'
#           ⚠️ f-string 안에서 중괄호를 두 번 {{ }} 쓰면 변수가 아닌 글자가 들어갑니다. 한 번만!
#       - 지시: 누진 구조(구간별 세율 + 누진공제)를 정확히 적용,
#               과세표준 전체에 한 세율만 곱하지 말 것,
#               계산 과정을 단계별로 보이고 최종 산출세액을 굵게 표시
#       - 질문: question
#   - answer 뒤에 "\n(소득세법 제55조 세율표 근거 · 참고용)" 를 붙여 반환
#   - 테스트: "과세표준 2000만원이면 세금 얼마야?", "과세표준 6천만원 세금 계산해줘"
# ▶ 확인: 2000만원 → 약 174만원 이어야 정답 (120만원이면 누진 미적용 오답)
# ------------------------------------------------------------

### 셀 11 — 봇 ②: 법령봇 (RAG 재사용)

In [ ]:
# [문제 15] 봇 ② 법령봇 law_bot()
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: law_bot(question) -> str
#   - 문제 12의 simple_rag() 를 재사용하여 결과를 그대로 반환
#   - 테스트: "과세기간은 어떻게 되나요?" 를 "📖 …" 형식으로 출력
# ------------------------------------------------------------

### 셀 12 — 봇 ③: 상담봇 (검색 없이 응대)

In [ ]:
# [문제 16] 봇 ③ 상담봇 chat_bot()
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: chat_bot(question) -> str
#   - 검색 없이 llm.invoke(프롬프트).content 를 반환
#     프롬프트 구성:
#       - "당신은 위니브 세무 상담 챗봇입니다. 1~2문장으로 친절히 답하되,
#          세금 계산·법령 질문이면 '계산/법령 기능을 이용해 주세요'라고 안내하세요."
#       - "손님: " + question
#   - 테스트: "안녕하세요, 뭘 도와주나요?" 를 "💬 …" 형식으로 출력
# ------------------------------------------------------------

# 6️⃣ 단계 — LangGraph 라우팅

질문을 보고 어느 봇에 보낼지 정하는 **라우터**를
만들고, LangGraph로 3개 봇을 연결합니다.

### 셀 13 — 라우터 (질문 분류)

In [ ]:
# [문제 17] 라우터 route() — 질문을 calc / law / chat 으로 분류
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: route(question) -> str   ("calc" | "law" | "chat")
#   - llm 에게 한 단어 분류를 요청하고 결과를 .content.lower() 로 소문자화 → decision
#     프롬프트에 분류 기준을 명시:
#       - law  : 소득세법 조문·규정·세율 구조·과세표준이 '무엇인지/어떻게 정해지는지' 설명 요청
#       - calc : 구체적 숫자가 주어져 '실제 세액 계산' 요청
#       - chat : 인사·잡담
#       - 마지막 줄 "질문: <question>\n분류:"
#   - 안전 파싱: ["calc", "law", "chat"] 를 순회하며 decision 안에 포함된 key 를 찾으면 return
#   - 아무것도 못 찾으면 "chat" 반환
#   - 테스트 출력
#       "과세표준은 어떻게 돼?"           → law
#       "과세표준 2000만원이면 세금 얼마야?" → calc
#       "안녕하세요"                      → chat
# ------------------------------------------------------------

### 셀 14 — LangGraph 조립

State → 라우터 → 조건부 분기로 3개 봇 연결.
(LangGraph + 조건부 엣지)

In [ ]:
# [문제 18] LangGraph 조립 — 라우터 → 3봇 분기
# ------------------------------------------------------------
# 요구사항
#   - typing 에서 TypedDict, langgraph.graph 에서 StateGraph, START, END import
#   (1) 상태 클래스 State(TypedDict): question: str, kind: str, answer: str
#   (2) 노드 함수 4개 (상태 s 를 받아 갱신할 키만 딕셔너리로 반환)
#       node_route : {"kind":   route(질문)}
#       node_calc  : {"answer": calc_bot(질문)}
#       node_law   : {"answer": law_bot(질문)}
#       node_chat  : {"answer": chat_bot(질문)}
#   (3) StateGraph(State) 생성 → 변수 graph
#       노드 등록(add_node): "router", "calc", "law", "chat"
#   (4) 엣지 연결
#       - START → "router"  (add_edge)
#       - "router" 에서 조건부 분기 (add_conditional_edges)
#           분기 함수: 상태의 "kind" 값을 반환하는 lambda
#           매핑: {"calc": "calc", "law": "law", "chat": "chat"}
#       - "calc", "law", "chat" → END
#   (5) 컴파일 → 변수 chatbot
#   - "✅ 멀티 챗봇 완성!" 출력
# ------------------------------------------------------------

### 셀 15 — 그래프 구조 보기

In [ ]:
# [문제 19] 그래프 구조 시각화
# ------------------------------------------------------------
# 요구사항
#   - try 블록: IPython.display 의 Image, display 를 import 하고
#       chatbot.get_graph().draw_mermaid_png() 결과를 Image 로 감싸 display
#   - except Exception 블록: 대신 chatbot.get_graph().draw_ascii() 를 print
# ------------------------------------------------------------

### 셀 16 — 완성된 챗봇 사용

In [ ]:
# [문제 20] 완성된 챗봇 사용 함수 ask()
# ------------------------------------------------------------
# 요구사항
#   - 함수 시그니처: ask(question)
#   - chatbot.invoke({"question": question, "kind": "", "answer": ""}) → result
#   - result["kind"] 에 따라 배지 선택 (딕셔너리 .get, 기본값 "🤖")
#       calc → "🧮 계산봇", law → "📖 법령봇", chat → "💬 상담봇"
#   - "🙋 질문" / "<배지>: 답변" + 빈 줄 형식으로 출력
#   - 아래 4개 질문으로 호출하여 각기 다른 봇이 답하는지 확인
#       "안녕하세요, 무엇을 도와주나요?"
#       "과세표준 3000만원에 15% 세율이면 세금은?"
#       "기본공제는 1인당 얼마까지 인정되나요?"
#       "종합소득세 과세기간은 어떻게 정해지나요?"
# ------------------------------------------------------------